In [ ]:
import pandas as pd
import numpy as np
import iqplot
from plot_tools import *
import itertools as it
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from scipy.spatial.distance import jensenshannon
hv.extension('bokeh')


In [ ]:
df_abundance = pd.read_csv('e003_round4_assembly_abundances.csv')
df_abundance_coalescence = pd.read_csv('e004_abundances_no_dups.csv')
df_abundance_coalescence['exp_type'].unique()
df_abundance.loc[df_abundance['exp_type']=='e003GlycerolRevival','passage']=df_abundance.loc[df_abundance['exp_type']=='e003GlycerolRevival','passage']+5

In [ ]:
def ecdf_transform_no_norm(data):
    return data.rank(method="first",ascending=False) #/ len(data)

def make_rank_abundance_assembly(mesocosm):
    subject,media = mesocosm.split('-')
    AAmGAM = df_abundance.loc[(df_abundance['type_mesocosm']==mesocosm)*(df_abundance['passage']==5),:]
    AAmGAM = AAmGAM.loc[AAmGAM['relative_abundance']>0,:]
    AAmGAM["rel abun ECDF"] = AAmGAM.groupby("sample")[
        "relative_abundance"
    ].transform(ecdf_transform_no_norm)
    fecal = df_abundance.loc[df_abundance['type_mesocosm']==f'{subject}-fecal',:]
    fecal= fecal.loc[fecal['relative_abundance']>0,:]
    fecal["rel abun ECDF"] = fecal.groupby("sample")[
        "relative_abundance"
    ].transform(ecdf_transform_no_norm)
    
    p4 = hv.Curve(
        data=fecal.sort_values(by='rel abun ECDF'),
        vdims=['relative_abundance','sample'],
        kdims=[('rel abun ECDF', 'ECDF'), ],label='Fecal',
    ).opts(width=250,height=300, color= bokeh.palettes.Set2[8][-2],#xlabel='Relative Abundance',
           logy=True,logx=True,ylim=(1e-3,1))
    for i,meso in enumerate(AAmGAM['mesocosm'].unique()):
        if i == 0:
            p1 = hv.Curve(
                data=AAmGAM.loc[AAmGAM['mesocosm']==meso,:].sort_values(by='rel abun ECDF'),
                vdims=['relative_abundance','sample'],
                kdims=[('rel abun ECDF', 'ECDF'), ],label=media,
            ).opts(width=200,height=200, color=  bokeh.palettes.Set2[8][0],#xlabel='Relative Abundance',
                   logy=True,logx=True,)
        else:
            p1 = hv.Curve(
                data=AAmGAM.loc[AAmGAM['mesocosm']==meso,:].sort_values(by='rel abun ECDF'),
                vdims=['relative_abundance','sample'],
                kdims=[('rel abun ECDF', 'ECDF'), ],
            ).opts(width=200,height=200, color=  bokeh.palettes.Set2[8][0],#xlabel='Relative Abundance',
                   logy=True,logx=True,)
        p4 = p4*p1
        #  ylabel='1-P(f>x)')
    
    return p4

In [ ]:
p = hv.render(make_rank_abundance_assembly('AA-mGAM'))
p.title.text = 'AA-mGAM'
p.yaxis.axis_label='Relative Abundance'
p.xaxis.axis_label='Species Rank'
bokeh.io.show(p)